In [1]:
import os
import joblib
import pandas as pd

from sqlalchemy import create_engine
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score
from supabase import create_client, Client

/Users/athachcha/Documents/GitHub/anaconda-Senior/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [36]:
from supabase import create_client, Client

url: str = "https://sqastdkrowpwfactzncc.supabase.co"
key: str = "sb_publishable_t1d7FzyLlzff1w_dYao4BQ_IgNxNcTW"

supabase: Client = create_client(url, key)

try:
    test = supabase.table("auth_user").select("id").execute()
    print("Connection successful >", test.data)
except Exception as e:
    print("Connect failed", e)

Connection successful > [{'id': 2}, {'id': 3}, {'id': 4}, {'id': 5}, {'id': 6}, {'id': 7}, {'id': 8}, {'id': 9}, {'id': 10}, {'id': 11}, {'id': 12}, {'id': 13}, {'id': 14}, {'id': 15}, {'id': 16}, {'id': 17}, {'id': 18}, {'id': 19}, {'id': 20}, {'id': 21}, {'id': 22}, {'id': 23}, {'id': 24}, {'id': 25}, {'id': 26}, {'id': 27}, {'id': 28}, {'id': 29}, {'id': 30}, {'id': 31}, {'id': 32}, {'id': 33}, {'id': 34}, {'id': 35}, {'id': 36}, {'id': 37}, {'id': 38}, {'id': 39}, {'id': 40}, {'id': 41}, {'id': 42}, {'id': 43}, {'id': 44}, {'id': 45}, {'id': 46}, {'id': 47}, {'id': 48}, {'id': 49}, {'id': 50}, {'id': 51}, {'id': 1}]


In [37]:
response = supabase.table("chemical_usage").select(
    "usage_id, inv_id, user_id, usage_date, value_use"
).execute()

df = pd.DataFrame(response.data)

In [38]:
response = supabase.table("chemical_usage").select("*").execute()
df = pd.DataFrame(response.data)

print(df.head())
print(df.columns)

    usage_id     inv_id  user_id                        usage_date  value_use
0  USEYVIW39  INVVH6M8B        9  2025-09-01T04:53:56.594219+00:00      16.55
1  USEVQ48Q1  INV9N3DKC       27  2025-07-19T13:20:10.778464+00:00      28.20
2  USEMSNL22  INV5JAP95       17  2025-05-21T02:53:33.207155+00:00      24.55
3  USE3M0TP9  INVY43WVL       46  2026-01-01T00:25:15.444424+00:00       6.75
4  USET3V4IJ  INVWHIWF2       38  2025-09-02T11:37:39.521973+00:00      36.56
Index(['usage_id', 'inv_id', 'user_id', 'usage_date', 'value_use'], dtype='object')


In [39]:
print(df.columns.tolist())

['usage_id', 'inv_id', 'user_id', 'usage_date', 'value_use']


In [40]:
df = df.dropna(subset=["usage_date", "value_use"])

print("Loaded data:", df.shape)
print(df.head())

Loaded data: (300, 5)
    usage_id     inv_id  user_id                        usage_date  value_use
0  USEYVIW39  INVVH6M8B        9  2025-09-01T04:53:56.594219+00:00      16.55
1  USEVQ48Q1  INV9N3DKC       27  2025-07-19T13:20:10.778464+00:00      28.20
2  USEMSNL22  INV5JAP95       17  2025-05-21T02:53:33.207155+00:00      24.55
3  USE3M0TP9  INVY43WVL       46  2026-01-01T00:25:15.444424+00:00       6.75
4  USET3V4IJ  INVWHIWF2       38  2025-09-02T11:37:39.521973+00:00      36.56


In [43]:
df["usage_date"] = pd.to_datetime(df["usage_date"], errors="coerce")
df = df.dropna(subset=["usage_date", "value_use", "inv_id", "user_id"])

df["year"] = df["usage_date"].dt.year
df["month"] = df["usage_date"].dt.month
df["day"] = df["usage_date"].dt.day
df["day_of_week"] = df["usage_date"].dt.dayofweek
df["hour"] = df["usage_date"].dt.hour

In [44]:
inv_encoder = LabelEncoder()
df["inv_id_encoded"] = inv_encoder.fit_transform(df["inv_id"])

In [45]:
features = [
    "inv_id_encoded",
    "user_id",
    "year",
    "month",
    "day",
    "day_of_week",
    "hour"
]

target = "value_use"

X = df[features]
y = df[target]

In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


In [47]:
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    max_depth=10
)

model.fit(X_train, y_train)


RandomForestRegressor(max_depth=10, n_estimators=200, random_state=42)

In [48]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Training completed")
print(f"MAE: {mae:.2f}")
print(f"R2 Score: {r2:.2f}")

Training completed
MAE: 11.33
R2 Score: -0.25


In [49]:
os.makedirs("ai_models", exist_ok=True)

joblib.dump(model, "ai_models/chemical_usage_model.pkl")
joblib.dump(inv_encoder, "ai_models/inv_encoder.pkl")

print("Model and encoder saved successfully")

Model and encoder saved successfully
